# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/demo/DAY_2_DEMO_SESSION_1_arxiv_mcp_langextract_pydantic.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





# Demo: arXiv MCP → LangExtract → Pydantic (before / after validation)

This notebook ties three pieces together the way you would in an agent stack:

1. **arXiv MCP server** — search and read papers through MCP tools (`search_papers`, `download_paper`, `read_paper`, …), same mental model as in Cursor. We connect with **`langchain-mcp-adapters`** + **stdio** (`uv tool run arxiv-mcp-server`).
2. **LangExtract** — turn the paper text into **grounded extractions** (spans + attributes) using a small task definition + examples.
3. **Pydantic (contract layer)** — **without** enforcement (loose dicts / wrong shapes) versus **with** explicit models and `ValidationError`s that tell you whether the system may continue.

**LLM (LangExtract):** **[OpenRouter](https://openrouter.ai)** with LangExtract’s **OpenAI-compatible** backend ([Using OpenAI models](https://github.com/google/langextract#using-openai-models)): **`OPENROUTER_API_KEY`**, optional **`OPENROUTER_MODEL`** (default matches the Tavily demo), **`OPENROUTER_BASE_URL`** (default `https://openrouter.ai/api/v1`). Install **`langextract[openai]`** and use **`fence_output=True`** / **`use_schema_constraints=False`** as in the LangExtract docs.

**MCP fallback:** if `uv` or `arxiv-mcp-server` is not on your PATH, the notebook falls back to the **`arxiv`** PyPI package so you can still run the LangExtract + Pydantic story end-to-end.

> You're not transforming text, you're **transforming uncertainty into enforceable structure**: representation + contracts + validation.


## Prerequisites

- **arXiv MCP** (recommended): install the server with `uv tool install arxiv-mcp-server` and ensure `uv` is on your PATH. Configure storage, e.g. `ARXIV_STORAGE_PATH` or `--storage-path` (see [arxiv-mcp-server](https://github.com/blazickjp/arxiv-mcp-server)).
- **LangExtract + OpenRouter**: `pip install "langextract[openai]"` and **`OPENROUTER_API_KEY`** ([keys](https://openrouter.ai/keys)).
- After the first `pip install` cell, **restart the kernel** if imports fail.

Cursor-style MCP config (reference only):

```json
{
  "mcpServers": {
    "arxiv-mcp-server": {
      "command": "uv",
      "args": ["tool", "run", "arxiv-mcp-server", "--storage-path", "/path/to/paper/storage"]
    }
  }
}
```

In [ ]:
!pip install -q \
    "langextract[openai]>=1.0" \
    "openai>=1.40" \
    "arxiv>=2.1" \
    "pydantic>=2,<3" \
    "python-dotenv>=1.0,<2" \
    "langchain-mcp-adapters>=0.1.0" \
    "mcp>=1.2"

## Environment

In [ ]:
from __future__ import annotations

import json
import os
import re
import tempfile
from collections import defaultdict
from pathlib import Path
from typing import Any

from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-5.4-nano")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("OPENROUTER_API_KEY: ").strip()
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

ARXIV_STORAGE = Path(
    os.getenv("ARXIV_STORAGE_PATH", Path(tempfile.mkdtemp(prefix="arxiv_mcp_demo_")))
).expanduser()
ARXIV_STORAGE.mkdir(parents=True, exist_ok=True)

print("Paper storage (MCP --storage-path):", ARXIV_STORAGE)
print("LangExtract via OpenRouter — model:", OPENROUTER_MODEL)
print("LangExtract via OpenRouter — base URL:", OPENROUTER_BASE_URL)
print("OPENROUTER_API_KEY set:", bool(OPENROUTER_API_KEY))


def display_lx_visualization(doc, stem: str) -> None:
    """Save JSONL + HTML, then emit a single in-notebook viewer (iframe when small enough for Colab)."""
    import base64

    import langextract as lx
    from IPython.display import HTML, display

    out_dir = ARXIV_STORAGE / "langextract_demo"
    out_dir.mkdir(parents=True, exist_ok=True)
    jsonl_name = f"{stem}.jsonl"
    lx.io.save_annotated_documents([doc], output_name=jsonl_name, output_dir=str(out_dir))
    jsonl_path = out_dir / jsonl_name
    html_obj = lx.visualize(str(jsonl_path))
    payload = html_obj.data if hasattr(html_obj, "data") else str(html_obj)
    (out_dir / f"{stem}.html").write_text(payload, encoding="utf-8")

    caption = (
        f'<p style="font-size:0.9em;color:#444;margin:0 0 10px 0">'
        f"Saved <code>{jsonl_name}</code> and <code>{stem}.html</code> under "
        f"<code>langextract_demo/</code>.</p>"
    )
    b64 = base64.b64encode(payload.encode("utf-8")).decode("ascii")
    if len(b64) < 1_800_000:
        viewer = (
            f'<iframe width="100%" height="720" '
            f'src="data:text/html;charset=utf-8;base64,{b64}" '
            'sandbox="allow-scripts allow-downloads allow-same-origin allow-popups allow-popups-to-escape-sandbox" '
            'style="border:1px solid #ccc;border-radius:6px;"></iframe>'
        )
    else:
        viewer = payload
    display(HTML(caption + viewer))

Paper storage (MCP --storage-path): /tmp/arxiv_mcp_demo_pvcozrkk
LangExtract via OpenRouter — model: openai/gpt-5.4-nano
LangExtract via OpenRouter — base URL: https://openrouter.ai/api/v1
OPENROUTER_API_KEY set: True


# Fetch paper text — arXiv MCP (stdio) or `arxiv` fallback

`MultiServerMCPClient.get_tools()` is **async**. In Jupyter, use top-level **`await`** in this section.

We locate tools whose names contain **`search`** and **`paper`**, then **`download`** / **`read`** as available, because prefixed names differ by adapter version.

Note: some **`mcp`** / **`langchain-mcp-adapters`** versions reject a stdio **`timeout`** argument; this demo omits it so MCP can start. If the MCP path still fails, the **`arxiv`** package fallback runs automatically.

In [ ]:
def _pick_tool(tools, *needles: str):
    for t in tools:
        name = t.name.lower()
        if all(n in name for n in needles):
            return t
    return None


def _tool_out_to_str(payload: Any) -> str:
    if payload is None:
        return ""
    if isinstance(payload, str):
        return payload
    if isinstance(payload, list):
        parts: list[str] = []
        for block in payload:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(str(block.get("text", "")))
            else:
                parts.append(str(block))
        return "\n".join(parts)
    return json.dumps(payload, default=str)


def _norm_arxiv_id(raw: str) -> str:
    s = (raw or "").strip()
    s = re.sub(r"^arxiv:", "", s, flags=re.I)
    if "arxiv.org" in s:
        s = s.split("/abs/")[-1]
    return s.split("v")[0].strip()


async def fetch_paper_via_mcp(query: str, max_results: int = 3) -> dict[str, Any]:
    from langchain_mcp_adapters.client import MultiServerMCPClient

    client = MultiServerMCPClient(
        {
            "arxiv": {
                "transport": "stdio",
                "command": "uv",
                "args": [
                    "tool",
                    "run",
                    "arxiv-mcp-server",
                    "--storage-path",
                    str(ARXIV_STORAGE),
                ],
            }
        },
        tool_name_prefix=True,
    )
    tools = await client.get_tools()
    search = _pick_tool(tools, "search", "paper")
    if search is None:
        raise RuntimeError(f"No search tool in {[t.name for t in tools]}")
    search_payload = {"query": query, "max_results": max_results}
    listed = await search.ainvoke(search_payload)
    text_out = _tool_out_to_str(listed)
    m = re.search(r"(\d{4}\.\d{4,5}|[a-z-]+(?:\.[A-Z]{2})?/\d{7})", text_out)
    if not m:
        raise RuntimeError(f"Could not parse arXiv id from search output: {text_out[:500]!r}")
    paper_id = _norm_arxiv_id(m.group(1))
    dl = _pick_tool(tools, "download", "paper") or _pick_tool(tools, "download")
    if dl is not None:
        await dl.ainvoke({"paper_id": paper_id})
    reader = _pick_tool(tools, "read", "paper")
    if reader is None:
        raise RuntimeError(f"No read_paper tool in {[t.name for t in tools]}")
    body = await reader.ainvoke({"paper_id": paper_id})
    body_str = _tool_out_to_str(body)
    return {
        "source": "arxiv_mcp",
        "paper_id": paper_id,
        "title": "",
        "body": body_str,
        "raw_search_excerpt": text_out[:1200],
    }


def fetch_paper_via_arxiv_pkg(query: str) -> dict[str, Any]:
    import arxiv as arxiv_pkg

    client = arxiv_pkg.Client()
    search = arxiv_pkg.Search(query=query, max_results=1)
    paper = next(client.results(search))
    pid = _norm_arxiv_id(paper.entry_id)
    title = " ".join((paper.title or "").split())
    summary = " ".join((paper.summary or "").split())
    body = f"Title: {title}\n\nAbstract:\n{summary}"
    return {
        "source": "arxiv_pypi",
        "paper_id": pid,
        "title": title,
        "body": body,
        "pdf_url": str(paper.pdf_url),
    }


QUERY = os.getenv("ARXIV_QUERY", "agentic workflow large language model")
paper_bundle: dict[str, Any] | None = None
mcp_err: str | None = None
try:
    paper_bundle = await fetch_paper_via_mcp(QUERY, max_results=3)
except Exception as e:
    mcp_err = repr(e)
    paper_bundle = fetch_paper_via_arxiv_pkg(QUERY)

if mcp_err:
    print("MCP path failed (using arxiv package). Reason:", mcp_err)
print("Source:", paper_bundle["source"], "| id:", paper_bundle["paper_id"])
print(paper_bundle["body"][:800] + ("…" if len(paper_bundle["body"]) > 800 else ""))

MCP path failed (using arxiv package). Reason: UnsupportedOperation('fileno')
Source: arxiv_pypi | id: 2504.19565
Title: Knowledge-Driven Agentic Scientific Corpus Distillation Framework for Biomedical Large Language Models Training

Abstract:
Corpus distillation for biomedical large language models (LLMs) seeks to address the pressing challenge of insufficient quantity and quality in open-source annotated scientific corpora, which remains a bottleneck for effective LLM training in biomedical research. This paper proposes a knowledge-driven, agentic framework for scientific corpus distillation, tailored explicitly for LLM training in the biomedical domain, addressing the challenge posed by the complex hierarchy of biomedical knowledge. Central to our approach is a collaborative multi-agent architecture, where specialized agents, each guided by the Medical Subject Headings (MeSH) hierarchy, work in con…


# LangExtract — structured extractions from the paper text

We define **classes** (`contribution`, `method`, `limitation`, `keyword`) and a **single-shot example** so the model stays close to **verbatim spans**. For §3–§4 we prefer **grounded** rows (`char_interval` is not `None`); if the model returns none, we fall back to **all** extractions so the Pydantic demo still has rows to merge.

OpenRouter ids like **`openai/gpt-4o-mini`** do not match LangExtract’s built-in `gpt-*` router regex, so we construct **`OpenAILanguageModel`** and pass **`model=`** into `lx.extract` (avoids `resolve_provider("openai")` issues). Per LangExtract: **`fence_output=True`**, **`use_schema_constraints=False`**.

In [ ]:
import textwrap

import langextract as lx

try:
    from langextract.providers.openai import OpenAILanguageModel
except ImportError as e:
    raise ImportError(
        'Install OpenAI deps: pip install "langextract[openai]" openai — then restart the kernel.'
    ) from e

PAPER_TEXT = paper_bundle["body"]
if len(PAPER_TEXT) > 12000:
    PAPER_TEXT = PAPER_TEXT[:12000] + "\n\n[… truncated for demo …]"

lx_prompt = textwrap.dedent(
    """\
    Extract research paper elements in order of appearance in the text.
    Use exact phrases from the paper for extraction_text (no paraphrase).
    Classes:
    - contribution: claimed novelty or main results
    - method: approach, model, dataset, or experimental setup (short span)
    - limitation: caveats, failure modes, or future work constraints
    - keyword: important technical term or acronym as written
    Add short attributes where useful, e.g. facet="architecture" or facet="dataset".
    """
)

examples = [
    lx.data.ExampleData(
        text=(
            "We introduce GraphTune, a two-stage method that first builds a task graph. "
            "Our key contribution is a 23% reduction in planning errors on ALFRED. "
            "Limitation: we assume static object sets. Keywords: LLM, planning."
        ),
        extractions=[
            lx.data.Extraction(
                extraction_class="method",
                extraction_text="two-stage method that first builds a task graph",
                attributes={"facet": "approach"},
            ),
            lx.data.Extraction(
                extraction_class="contribution",
                extraction_text="a 23% reduction in planning errors on ALFRED",
                attributes={"facet": "result"},
            ),
            lx.data.Extraction(
                extraction_class="limitation",
                extraction_text="we assume static object sets",
                attributes={"facet": "assumption"},
            ),
            lx.data.Extraction(
                extraction_class="keyword",
                extraction_text="LLM",
                attributes={"facet": "acronym"},
            ),
            lx.data.Extraction(
                extraction_class="keyword",
                extraction_text="planning",
                attributes={"facet": "topic"},
            ),
        ],
    )
]

lx_result = None
if OPENROUTER_API_KEY:
    lx_lm = OpenAILanguageModel(
        model_id=OPENROUTER_MODEL,
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
    )
    lx_result = lx.extract(
        text_or_documents=PAPER_TEXT,
        prompt_description=lx_prompt,
        examples=examples,
        model=lx_lm,
        fence_output=True,
        use_schema_constraints=False,
        extraction_passes=1,
        max_workers=4,
    )
    grounded = [e for e in lx_result.extractions if e.char_interval]
    print("Total extractions:", len(lx_result.extractions), "| grounded:", len(grounded))
else:
    print("Skip LangExtract call — set OPENROUTER_API_KEY (environment cell).")

LangExtract: model=openai/gpt-5.4-nano, current=1,721 chars, processed=0 chars:  [00:08]

Total extractions: 12 | grounded: 12


# LangExtract — evidence-only vs optional world knowledge (**same paper as §2**)

The [LangExtract README](https://github.com/google/langextract) uses a literary snippet to make the point: attributes can stay **tight to wording**, or you can **invite** extra attributes from the model’s **world knowledge**—controlled by **prompt + examples**.

Here we do that on the **actual arXiv body** from §1 (truncated like §2), with the **same four classes** as §2 (`contribution`, `method`, `limitation`, `keyword`).

- **Run A** — attributes should be **defensible from the paper text** only.
- **Run B** — same verbatim spans, but the prompt **allows** optional `background` (or similar) from general domain knowledge, marked with `attribute_source="inferred"`; text-grounded attrs use `attribute_source="text"`.

Both runs share **one** few-shot example (paper-style) so the model keeps producing parseable `extractions` JSON; only the **instruction** paragraph changes.

The interactive HTML viewer for extractions appears **once** in **§5** (after the markdown there), using the §2 run so the notebook stays uncluttered.

In [ ]:
import textwrap

import langextract as lx
from langextract.providers.openai import OpenAILanguageModel

if "paper_bundle" not in globals():
    raise RuntimeError("Run §1 first so paper_bundle is defined.")

PAPER_COMPARE_TEXT = paper_bundle["body"]
if len(PAPER_COMPARE_TEXT) > 12000:
    PAPER_COMPARE_TEXT = PAPER_COMPARE_TEXT[:12000] + "\n\n[… truncated for demo …]"

paper_compare_examples = [
    lx.data.ExampleData(
        text=(
            "We introduce BioDistill, a MeSH-guided multi-agent pipeline for biomedical corpus distillation. "
            "On MedQA we report accuracy gains over strong baselines. "
            "Limitation: we assume annual MeSH releases. Keywords: LLM, MeSH."
        ),
        extractions=[
            lx.data.Extraction(
                extraction_class="method",
                extraction_text="MeSH-guided multi-agent pipeline for biomedical corpus distillation",
                attributes={"facet": "approach", "attribute_source": "text"},
            ),
            lx.data.Extraction(
                extraction_class="contribution",
                extraction_text="accuracy gains over strong baselines",
                attributes={"facet": "result", "attribute_source": "text"},
            ),
            lx.data.Extraction(
                extraction_class="limitation",
                extraction_text="we assume annual MeSH releases",
                attributes={"facet": "assumption", "attribute_source": "text"},
            ),
            lx.data.Extraction(
                extraction_class="keyword",
                extraction_text="LLM",
                attributes={"facet": "acronym", "attribute_source": "text"},
            ),
            lx.data.Extraction(
                extraction_class="keyword",
                extraction_text="MeSH",
                attributes={"facet": "resource", "attribute_source": "text"},
            ),
        ],
    )
]

prompt_a_evidence = textwrap.dedent(
    """\
    Extract research paper elements in order of appearance from the document.
    Use exact phrases from the paper for extraction_text (no paraphrase).
    Classes: contribution, method, limitation, keyword (same as §2).
    For every extraction, attributes must be justified by the paper wording only.
    Use attribute_source="text" on each attribute you output.
    Respond with valid JSON in the required format (including an extractions array).
    """
)

prompt_b_hybrid = textwrap.dedent(
    """\
    Extract research paper elements in order of appearance from the document.
    Use exact phrases from the paper for extraction_text (no paraphrase).
    Classes: contribution, method, limitation, keyword (same as §2).
    You may add an optional attribute background with general domain context not spelled out
    in the paper; when you do, set attribute_source="inferred" for that key only.
    All other attributes must be grounded in the paper and use attribute_source="text".
    You must still return valid JSON in the required format with an extractions array.
    """
)

if OPENROUTER_API_KEY:
    lx_lm_cmp = OpenAILanguageModel(
        model_id=OPENROUTER_MODEL,
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
    )
    _kw = dict(
        text_or_documents=PAPER_COMPARE_TEXT,
        examples=paper_compare_examples,
        model=lx_lm_cmp,
        fence_output=True,
        use_schema_constraints=False,
        extraction_passes=1,
        max_workers=4,
        max_char_buffer=1200,
        temperature=0.0,
    )

    print("=== A) Evidence-only attributes (same paper text as §2) ===")
    paper_ev = lx.extract(prompt_description=prompt_a_evidence, **_kw)
    for e in paper_ev.extractions[:20]:
        print(e.extraction_class, "|", e.extraction_text[:120], "|", dict(e.attributes or {}))

    print("\n=== B) Same paper; optional inferred background allowed ===")
    paper_hy = None
    try:
        paper_hy = lx.extract(prompt_description=prompt_b_hybrid, **_kw)
    except Exception as ex:
        print("Run B failed (model returned unparsable JSON). Error:", ex)
    else:
        for e in paper_hy.extractions[:20]:
            print(e.extraction_class, "|", e.extraction_text[:120], "|", dict(e.attributes or {}))
else:
    print("Set OPENROUTER_API_KEY to run the paper comparison.")

=== A) Evidence-only attributes (same paper text as §2) ===


LangExtract: model=openai/gpt-5.4-nano, current=1,721 chars, processed=0 chars:  [00:03]


contribution | Corpus distillation for biomedical large language models (LLMs) seeks to address the pressing challenge of insufficient  | {'facet': 'problem', 'attribute_source': 'text'}
contribution | This paper proposes a knowledge-driven, agentic framework for scientific corpus distillation, tailored explicitly for LL | {'facet': 'proposal', 'attribute_source': 'text'}
method | Central to our approach is a collaborative multi-agent architecture, where specialized agents, each guided by the Medica | {'facet': 'architecture_and_pipeline', 'attribute_source': 'text'}
method | This agentic framework collectively generates and refines domain-specific question-answer pairs | {'facet': 'generation_and_refinement', 'attribute_source': 'text'}
method | ensuring comprehensive coverage and consistency with biomedical ontologies while minimizing manual involvement | {'facet': 'ensures_properties', 'attribute_source': 'text'}
keyword | biomedical large language models (LLMs) | {'facet': 'domain'

LangExtract: model=openai/gpt-5.4-nano, current=1,721 chars, processed=0 chars:  [00:04]

contribution | proposes a knowledge-driven, agentic framework for scientific corpus distillation, tailored explicitly for LLM training  | {'facet': 'framework', 'attribute_source': 'text'}
method | Central to our approach is a collaborative multi-agent architecture, where specialized agents, each guided by the Medica | {'facet': 'architecture', 'attribute_source': 'text'}
method | This agentic framework collectively generates and refines domain-specific question-answer pairs | {'facet': 'generation process', 'attribute_source': 'text'}
contribution | ensuring comprehensive coverage and consistency with biomedical ontologies while minimizing manual involvement | {'facet': 'benefit', 'attribute_source': 'text'}
keyword | Corpus distillation for biomedical large language models (LLMs) | {'facet': 'task', 'attribute_source': 'text'}
keyword | biomedical large language models (LLMs) | {'facet': 'domain model', 'attribute_source': 'text'}
keyword | Medical Subject Headings (MeSH) | {'facet':

# Before Pydantic — loose dicts

Typical failure modes when you **skip** schema enforcement:

- **Wrong container types** (a single string where downstream code expects a **list**).
- **Silent drops** (`None` ids, empty title) because nothing validates required fields.
- **Heterogeneous items** (e.g. an **integer** in a `list[str]`—here we use **`42` on purpose**, not an HTTP status) that break serializers or UI code later.

Below we **intentionally** build a sloppy `"analysis"` object from the same extractions. The bogus int is appended **only when** there is at least one real `method` span, so an empty LangExtract run does not look like a “404 error”.

The tiny **`naive_agent`** at the end counts `methods` without checking types: wrong shapes often **fail silently** or return nonsense. **§4** shows the same payload rejected **explicitly** by a contract—contrast that with "just formatting."


In [ ]:
from pydantic import BaseModel, Field, ValidationError, field_validator


def _demo_extractions():
    if lx_result is not None:
        grounded_only = [e for e in lx_result.extractions if e.char_interval]
        return grounded_only if grounded_only else list(lx_result.extractions)
    class E:
        def __init__(self, cls, text, attrs=None, char_interval=True):
            self.extraction_class = cls
            self.extraction_text = text
            self.attributes = attrs or {}
            self.char_interval = char_interval

    return [
        E("contribution", "We propose a retrieval-augmented agent framework."),
        E("method", "We evaluate on HotpotQA and BFCL."),
        E("limitation", "Our approach adds latency due to tool calls."),
        E("keyword", "LangGraph"),
    ]


def build_loose_analysis(
    paper_id: str, title: str, extractions
) -> dict[str, Any]:
    """Naive merge — easy to write, hard to trust."""
    contributions: list[Any] = []
    methods: list[Any] = []
    limitations: list[Any] = []
    keywords: list[Any] = []
    for ex in extractions:
        cls_key = (ex.extraction_class or "").strip().lower()
        bucket = {
            "contribution": contributions,
            "method": methods,
            "limitation": limitations,
            "keyword": keywords,
        }.get(cls_key)
        if bucket is None:
            continue
        bucket.append(ex.extraction_text)
    methods_out = list(methods)
    if methods_out:
        methods_out.append(42)
    return {
        "paper_id": None,
        "title": title or "unknown",
        "contributions": "; ".join(contributions) if contributions else "",
        "methods": methods_out,
        "limitations": limitations,
        "keywords": ", ".join(keywords),
    }


exts = _demo_extractions()
loose = build_loose_analysis(paper_bundle["paper_id"], paper_bundle.get("title", ""), exts)
print(json.dumps(loose, indent=2, default=str))


def naive_agent(loose: dict):
    try:
        return len(loose["methods"])
    except:
        return "🤷 silently wrong"


print("Naive agent:", naive_agent(loose))

{
  "paper_id": null,
  "title": "Knowledge-Driven Agentic Scientific Corpus Distillation Framework for Biomedical Large Language Models Training",
  "contributions": "autonomously extract, synthesize, and self-evaluate high-quality textual data; tailored explicitly for LLM training in the biomedical domain",
  "methods": [
    "a knowledge-driven, agentic framework for scientific corpus distillation",
    "a collaborative multi-agent architecture",
    "specialized agents",
    "each guided by the Medical Subject Headings (MeSH) hierarchy",
    42
  ],
  "limitations": [
    "insufficient quantity and quality in open-source annotated scientific corpora",
    "the challenge posed by the complex hierarchy of biomedical knowledge"
  ],
  "keywords": "scientific corpus distillation, biomedical large language models (LLMs), Medical Subject Headings (MeSH), LLM training"
}
Naive agent: 5


## 4) Contract Enforcement Layer (Pydantic)

Same extractions, but we treat the analysis shape as a **system contract**, not “nice-to-have typing”:

1. **Group** by `extraction_class` deterministically.
2. **Require** `paper_id` and `title`.
3. Use **`field_validator`** to coerce a comma-separated keyword string into `list[str]` if needed.
4. Let Pydantic **reject** the loose dict from §3 with an explicit **`ValidationError`** (vs. silent corruption).

> **Demo line:** MCP gives you access to information. LangExtract gives you structure. Pydantic decides whether your system is allowed to continue.

**Stack framing:** many pipelines look like MCP → agent → prompt → hope. A sturdier pattern is **MCP → extraction layer → contract layer → agent**—structure first, then enforce what downstream code may assume.

In [ ]:
class PaperAnalysis(BaseModel):
    paper_id: str = Field(..., min_length=4)
    title: str = Field(..., min_length=1)
    contributions: list[str] = Field(default_factory=list)
    methods: list[str] = Field(default_factory=list)
    limitations: list[str] = Field(default_factory=list)
    keywords: list[str] = Field(default_factory=list)

    @field_validator("keywords", mode="before")
    @classmethod
    def split_keywords(cls, v: Any) -> Any:
        if isinstance(v, str):
            parts = [p.strip() for p in re.split(r"[;,]", v) if p.strip()]
            return parts or v
        return v


def extractions_to_analysis(
    paper_id: str, title: str, extractions, body: str = ""
) -> PaperAnalysis:
    by: dict[str, list[str]] = defaultdict(list)
    for ex in extractions:
        cls_key = (ex.extraction_class or "").strip().lower()
        by[cls_key].append(ex.extraction_text.strip())
    title_clean = (title or "").strip()
    if not title_clean and body:
        m = re.search(r"Title:\s*(.+?)(?:\n|$)", body, re.I)
        title_clean = m.group(1).strip() if m else "(untitled)"
    return PaperAnalysis(
        paper_id=paper_id,
        title=title_clean,
        contributions=by.get("contribution", []),
        methods=by.get("method", []),
        limitations=by.get("limitation", []),
        keywords=by.get("keyword", []),
    )


def agent_decision(analysis: PaperAnalysis) -> str:
    if len(analysis.methods) == 0:
        return "❌ No method detected — cannot evaluate paper"

    if any("limitation" in l.lower() for l in analysis.limitations):
        return "⚠️ Proceed with caution"

    return "✅ Suitable for downstream use"


validated = extractions_to_analysis(
    paper_bundle["paper_id"],
    paper_bundle.get("title", ""),
    exts,
    body=paper_bundle["body"],
)
print(validated.model_dump_json(indent=2))
print(agent_decision(validated))

print("\n--- Trying to validate the *loose* dict from §3 ---")
try:
    PaperAnalysis.model_validate(loose)
except ValidationError as err:
    print(err)

{
  "paper_id": "2504.19565",
  "title": "Knowledge-Driven Agentic Scientific Corpus Distillation Framework for Biomedical Large Language Models Training",
  "contributions": [
    "autonomously extract, synthesize, and self-evaluate high-quality textual data",
    "tailored explicitly for LLM training in the biomedical domain"
  ],
  "methods": [
    "a knowledge-driven, agentic framework for scientific corpus distillation",
    "a collaborative multi-agent architecture",
    "specialized agents",
    "each guided by the Medical Subject Headings (MeSH) hierarchy"
  ],
  "limitations": [
    "insufficient quantity and quality in open-source annotated scientific corpora",
    "the challenge posed by the complex hierarchy of biomedical knowledge"
  ],
  "keywords": [
    "scientific corpus distillation",
    "biomedical large language models (LLMs)",
    "Medical Subject Headings (MeSH)",
    "LLM training"
  ]
}
✅ Suitable for downstream use

--- Trying to validate the *loose* dict from

# Repair loop (failure → fix → re-validate)

Production stacks often **retry** after a failed contract check: a tool or model **patches** the payload, then you validate again. Below is a **toy** repair: drop non-`str` **methods**, fill missing **`paper_id`** from **`paper_bundle`**, and turn string **`contributions`** into a **`list[str]`** so **`PaperAnalysis.model_validate`** can succeed on the §3 dict—showing **rejection**, **repair**, and **admission** in one arc.


In [ ]:
def repair_loose(loose: dict) -> dict:
    # Simulate LLM / tool repair after ValidationError
    fixed = {**loose}
    if isinstance(fixed.get("methods"), list):
        fixed["methods"] = [m for m in fixed["methods"] if isinstance(m, str)]
    fixed["paper_id"] = fixed.get("paper_id") or paper_bundle["paper_id"]
    if isinstance(fixed.get("contributions"), str):
        s = fixed["contributions"].strip()
        fixed["contributions"] = [s] if s else []
    return fixed


fixed = repair_loose(loose)
repaired = PaperAnalysis.model_validate(fixed)
print("After repair, contract passes — agent_decision:", agent_decision(repaired))
print(repaired.model_dump_json(indent=2))


After repair, contract passes — agent_decision: ✅ Suitable for downstream use
{
  "paper_id": "2504.19565",
  "title": "Knowledge-Driven Agentic Scientific Corpus Distillation Framework for Biomedical Large Language Models Training",
  "contributions": [
    "autonomously extract, synthesize, and self-evaluate high-quality textual data; tailored explicitly for LLM training in the biomedical domain"
  ],
  "methods": [
    "a knowledge-driven, agentic framework for scientific corpus distillation",
    "a collaborative multi-agent architecture",
    "specialized agents",
    "each guided by the Medical Subject Headings (MeSH) hierarchy"
  ],
  "limitations": [
    "insufficient quantity and quality in open-source annotated scientific corpora",
    "the challenge posed by the complex hierarchy of biomedical knowledge"
  ],
  "keywords": [
    "scientific corpus distillation",
    "biomedical large language models (LLMs)",
    "Medical Subject Headings (MeSH)",
    "LLM training"
  ]
}


#  LangExtract viewer (one embedded visualization)

**Why LangExtract is useful here:** it returns **structured extractions** (class + **verbatim span** + optional **attributes**) tied to **positions in the source text**, not a loose summary. That helps you **audit grounding** (“does this claim actually appear?”), **feed downstream tools** (RAG chunks, citation checks, structured DB rows), and **compare prompts** (same paper, different instructions—like §2 vs §2b) without losing traceability to the document.

The cell below writes JSONL and HTML under **`ARXIV_STORAGE/langextract_demo/`** and calls **`display_lx_visualization`** once for the **§2** extraction. It uses a **single** `HTML` output: a short file caption plus either a **base64 iframe** (works well in **Colab**) or inline HTML if the page is too large for an iframe data URL.

In [ ]:
if lx_result is not None:
    pid = paper_bundle["paper_id"].replace("/", "_")
    display_lx_visualization(lx_result, f"extractions_{pid}")
else:
    print("No lx_result — run LangExtract first.")

LangExtract: Saving to extractions_2504.19565.jsonl: 1 docs [00:00, 789.29 docs/s]

✓ Saved 1 documents to extractions_2504.19565.jsonl



LangExtract: Loading extractions_2504.19565.jsonl: 100%|██████████| 5.31k/5.31k [00:00<00:00, 8.44MB/s]

✓ Loaded 1 documents from extractions_2504.19565.jsonl
